In [ ]:
from __future__ import annotations
import pendulum
import requests
import logging
from airflow.models.dag import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.postgres.hooks.postgres import PostgresHook

# 로깅 설정
logger = logging.getLogger(__name__)

# DAG 정의
with DAG(
    dag_id="restcountries_to_redshift_full_refresh",
    description="Fetch country data from RESTCountries API and load to Redshift (Full Refresh)",
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    schedule_interval="30 6 * * 6",  # 매주 토요일 오전 6시 30분 (UTC)
    catchup=False,
    tags=["API", "Redshift", "FullRefresh"],
) as dag:

    def extract_and_load_countries(redshift_conn_id: str, redshift_schema: str, redshift_table: str):
        """
        RESTCountries API로부터 국가 정보를 가져와
        Redshift 테이블에 Full Refresh 방식으로 적재하는 함수
        """
        logger.info("🌍 Fetching data from RESTCountries API...")
        api_url = "https://restcountries.com/v3/all"

        response = requests.get(api_url)
        response.raise_for_status()  # HTTP 오류 시 예외 발생
        countries_data = response.json()
        logger.info(f"✅ API 호출 성공: {len(countries_data)}개 국가 데이터 수신")

        # Redshift 연결
        redshift_hook = PostgresHook(postgres_conn_id=redshift_conn_id)

        # 테이블 생성 (없으면 생성)
        create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {redshift_schema}.{redshift_table} (
            country VARCHAR(256),
            population BIGINT,
            area FLOAT
        );
        """
        redshift_hook.run(create_table_sql)
        logger.info(f"🗂️ 테이블 확인/생성 완료: {redshift_schema}.{redshift_table}")

        # Full Refresh 전략: 기존 데이터 삭제
        truncate_table_sql = f"TRUNCATE TABLE {redshift_schema}.{redshift_table};"
        redshift_hook.run(truncate_table_sql)
        logger.info(f"🧹 기존 데이터 삭제 완료 (Full Refresh 모드)")

        # 데이터 추출 및 삽입 준비
        rows_to_insert = []
        for country in countries_data:
            try:
                name = country.get("name", {}).get("official")
                population = country.get("population")
                area = country.get("area")

                if name and population is not None and area is not None:
                    rows_to_insert.append((name, population, area))
            except Exception as e:
                logger.warning(f"⚠️ Skipping invalid record: {e}")
                continue

        # 데이터 삽입
        if rows_to_insert:
            redshift_hook.insert_rows(
                table=f"{redshift_schema}.{redshift_table}",
                rows=rows_to_insert,
                target_fields=["country", "population", "area"]
            )
            logger.info(f"✅ {len(rows_to_insert)}건의 국가 데이터 Redshift 적재 완료.")
        else:
            logger.warning("⚠️ 삽입할 유효한 데이터가 없습니다.")

    # PythonOperator 정의
    countries_etl_task = PythonOperator(
        task_id="extract_countries_and_load_to_redshift",
        python_callable=extract_and_load_countries,
        op_kwargs={
            "redshift_conn_id": "redshift_dev_db",   # Airflow Connections ID
            "redshift_schema": "your_schema_name",   # 본인 스키마 이름
            "redshift_table": "countries_full_refresh"  # 생성할 테이블 이름
        },
    )


In [ ]:
from __future__ import annotations

import pendulum
import requests
import logging
from airflow.models.dag import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.postgres.hooks.postgres import PostgresHook
# Note: 'requests' 라이브러리는 Airflow 컨테이너에 설치되어 있어야 합니다.

logger = logging.getLogger(__name__)

# ⭐ 사용자 정의 변수 (필요에 따라 수정) ⭐
REDSHIFT_CONN_ID = "redshift_dev_db"  # Airflow Connection ID와 일치해야 함
REDSHIFT_SCHEMA = "your_schema_name"   # 본인의 Redshift 스키마 이름으로 변경
REDSHIFT_TABLE = "countries_full_refresh"


with DAG(
    dag_id="restcountries_to_redshift_full_refresh",
    # 2025년 1월 1일 UTC 시작 (future start_date 권장)
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    # 매주 토요일 오전 6시 30분 (UTC) 실행
    schedule="30 6 * * 6",
    catchup=False,
    tags=["API", "Redshift", "FullRefresh"],
    doc_md=__doc__,
) as dag:

    def extract_and_load_countries(conn_id, schema, table):
        """
        Restcountries API를 호출하여 데이터를 추출하고 Redshift에 Full Refresh 방식으로 적재하는 함수
        """
        api_url = "https://restcountries.com/v3/all"

        # 1. 데이터 추출 (Extract)
        logger.info(f"API 호출 시작: {api_url}")
        response = requests.get(api_url)
        response.raise_for_status() # HTTP 상태 코드가 4xx 또는 5xx일 경우 오류 발생

        countries_data = response.json()
        redshift_hook = PostgresHook(postgres_conn_id=conn_id)

        # 2. Redshift 테이블 준비 및 Full Refresh (Truncate)
        create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {schema}.{table} (
            country VARCHAR(256),
            population BIGINT,
            area FLOAT
        );
        """
        redshift_hook.run(create_table_sql)
        logger.info(f"테이블 {schema}.{table} 준비 완료.")

        # 기존 데이터 삭제 (Full Refresh 전략)
        redshift_hook.run(f"TRUNCATE TABLE {schema}.{table};")
        logger.info("기존 데이터 TRUNCATE 완료.")

        # 3. 데이터 변환 및 적재 준비 (Transform & Load)
        rows_to_insert = []
        for c in countries_data:
            try:
                # 필요한 3가지 필드 추출
                name = c.get("name", {}).get("official")
                pop = c.get("population")
                area = c.get("area")

                # 추출된 값이 모두 유효한 경우에만 적재 리스트에 추가
                if name and pop is not None and area is not None:
                    rows_to_insert.append((name, pop, area))
            except Exception as e:
                logger.warning(f"데이터 처리 중 오류 발생. 해당 레코드 건너뜀: {e}")
                continue

        if rows_to_insert:
            # 데이터베이스에 일괄 삽입 (Postgres Hook의 insert_rows 사용)
            redshift_hook.insert_rows(
                table=f"{schema}.{table}",
                rows=rows_to_insert,
                target_fields=["country", "population", "area"]
            )
            logger.info(f"✅ 총 {len(rows_to_insert)} 건의 레코드 적재 완료.")
        else:
            logger.warning("⚠️ 유효한 레코드가 없어 적재할 데이터가 없습니다.")


    countries_etl_task = PythonOperator(
        task_id="extract_countries_and_load_to_redshift",
        python_callable=extract_and_load_countries,
        op_kwargs={
            "conn_id": REDSHIFT_CONN_ID,
            "schema": REDSHIFT_SCHEMA,
            "table": REDSHIFT_TABLE
        },
    )